# 02 · First cable exercise: a COMPAS graph
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

A cable has **5 nodes** and **4 edges**. Nodes **0** and **4** are supports; nodes **1**, **2** and **3** carry downward point loads.

We store the cable as a COMPAS `Graph`, find its equilibrium with `compas_fd`, and display it with `compas_notebook.Viewer`, following the [official COMPAS JupyterLite graph example](https://compas.dev/compas-jupyterlite/lab/index.html?path=random-graph.ipynb).

Choose **Run → Run All Cells**, or use **Shift+Enter** cell by cell.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps
%pip install -q compas_fd==0.5.4


In [ ]:
# Enable interactive widgets when this notebook is opened in Google Colab.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
from compas.colors import Color
from compas.datastructures import Graph
from compas.geometry import Pointcloud
from compas_fd.solvers import fd_numpy
from compas_notebook import Viewer
from compas_notebook.config import Config


## 1. Define the nodes and edges
Each node has coordinates `(x, y, z)` in metres. An edge joins two node keys. These five nodes start on a straight line; the first and last nodes stay fixed.


In [ ]:
nodes = [
    [0, 0, 0],  # node 0: support
    [1, 0, 0],  # node 1: free
    [2, 0, 0],  # node 2: free
    [3, 0, 0],  # node 3: free
    [4, 0, 0],  # node 4: support
]
edges = [(0, 1), (1, 2), (2, 3), (3, 4)]
fixed = [0, 4]

cable = Graph.from_nodes_and_edges(nodes, edges)
print(cable)
for key in cable.nodes():
    print(f"Node {key}: {cable.node_coordinates(key)}")


## 2. View the starting graph
The COMPAS scene draws the graph's nodes and edges. Larger blue points mark the two supports. Orbit and zoom in the viewer to inspect the connectivity.


In [ ]:
config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False
config.view.camera.position = [2, -7, 2]
config.view.camera.target = [2, 0, -0.5]

initial_viewer = Viewer(config=config)
initial_viewer.scene.add(cable.copy(), color=Color.from_hex("#dc4b8d"), nodesize=0.15)
supports = Pointcloud([cable.node_coordinates(key) for key in fixed])
initial_viewer.scene.add(supports, color=Color.blue(), pointsize=0.25)
initial_viewer.show()


## 3. Find equilibrium with COMPAS FD
Apply a downward load `P` in kN to each free node. Set the same positive force density `q` in kN/m on every edge. Force density means **tension divided by segment length**.

`fd_numpy` computes the equilibrium coordinates, edge forces and residual forces. This example prescribes force densities; the total cable length is a result of the solve.


In [ ]:
P = 1.0  # downward load at each free node [kN]
q = 2.0  # force density for every edge [kN/m]

# Node keys are 0, 1, 2, 3, 4, matching their positions in the coordinate list.
vertices = cable.nodes_attributes("xyz")
edges = list(cable.edges())
loads = [[0, 0, 0] if key in fixed else [0, 0, -P] for key in cable.nodes()]

result = fd_numpy(
    vertices=vertices,
    fixed=fixed,
    edges=edges,
    forcedensities=[q] * cable.number_of_edges(),
    loads=loads,
)

# Put the computed coordinates back into the COMPAS graph.
for key, xyz in zip(cable.nodes(), result.vertices):
    cable.node_attributes(key, "xyz", xyz.tolist())

for key in cable.nodes():
    print(f"Node {key}: z = {cable.node_attribute(key, 'z'):.2f} m")
print(f"Midspan sag: {-cable.node_attribute(2, 'z'):g} m")


## 4. View and check the equilibrium cable
The scene uses the updated COMPAS graph. Residuals at the free nodes should be approximately zero; residuals at fixed nodes correspond to support reactions.


In [ ]:
viewer = Viewer(config=config)
viewer.scene.add(cable, color=Color.from_hex("#dc4b8d"), nodesize=0.15)
supports = Pointcloud([cable.node_coordinates(key) for key in fixed])
viewer.scene.add(supports, color=Color.blue(), pointsize=0.25)
viewer.show()

free = [key for key in cable.nodes() if key not in fixed]
max_residual = max(abs(value) for key in free for value in result.residuals[key])
assert max_residual < 1e-6, "A free node is not in equilibrium."
print(f"Largest free-node residual: {max_residual:.2e} kN")
print("All three free nodes are in equilibrium.")
for edge, force in zip(cable.edges(), result.forces):
    print(f"Edge {edge}: tension {float(force[0]):.3f} kN")


## 5. Try it yourself
1. Predict which free node will be lowest before solving.
2. Double `P` from 1 to 2 kN. What happens to the sag?
3. Restore `P = 1`, then double `q` from 2 to 4 kN/m. What changes?

After changing a parameter, rerun the solver and the equilibrium viewer cells.

**Starting result:** free-node heights **−0.75, −1.00, −0.75 m**, so midspan sag is **1 m**.

Use `/notebook cable` in BRG for instructions. Save with **Ctrl+S** (or **⌘S**) and download a copy through **File → Save and Export Notebook As → Notebook**.
